# Figure 5 panel E - residue-level forest plots

In [1]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set(style='white')

In [2]:
# --- Manuscript plotting style ---
highlight_color = 'cornflowerblue'
main_color = "#AAAAAA"
c_attention = '#378ADD'

axis_label_font_size = 7
tick_label_font_size = 6
plt.rcParams.update({
    'font.size': axis_label_font_size,
    'axes.titlesize': axis_label_font_size,
    'axes.labelsize': axis_label_font_size,
    'xtick.labelsize': tick_label_font_size,
    'ytick.labelsize': tick_label_font_size,
    'legend.fontsize': tick_label_font_size,
    'legend.title_fontsize': axis_label_font_size,
})

# --- Input paths ---
INTERMEDIATE_DIR = 'intermediate_data/'
COX_RES_DIR = os.path.join(INTERMEDIATE_DIR, 'panel_b_positional_cox_results')
MODEL_SUMMARY_DIR = os.path.join(INTERMEDIATE_DIR, 'panel_e_positional_cox_model_summaries')

# --- Output path ---
FIGURE_DIR = 'figures/'
os.makedirs(FIGURE_DIR, exist_ok=True)

# --- Analysis configuration ---
EXP_NAME = 'first_immuno_lot_only_no_bmi'
LOCUS_OF_INTEREST = 'DRB1'
TOX_TYPE = 'adrenal_insufficiency'
GRADES = [0, 2, 3]  # 0 = Grade 1+, 2 = Grade 2+, 3 = Grade 3+


In [3]:
def _forest_plot_from_summary(res, locus, pos, tox_type, ae_grade, exp_name,
                               logscale=True, hr_clip=(0.01, 100), figure_dir=FIGURE_DIR):
    """Draw & save the residue-only forest plot given an already-loaded
    Cox model summary DataFrame. Internal helper for `plot_residue_forest`
    below; this is Panel E's plotting logic"""
    LW = 0.5
    res = res.copy()

    res = res[res.index.str.contains('Residue', case=False)]
    if res.empty:
        return None

    res['exp(coef)'] = res['exp(coef)'].clip(*hr_clip)
    res['exp(coef) lower 95%'] = res['exp(coef) lower 95%'].clip(*hr_clip)
    res['exp(coef) upper 95%'] = res['exp(coef) upper 95%'].clip(*hr_clip)
    res['is_sig'] = np.where(res['p'] < 0.05, 'P < 0.05', 'NS')
    res = res.sort_values('exp(coef)', ascending=True)

    yticklabels = [
        ele.split('T.')[-1].split(']')[0]
           .replace('_', ' ').title()
           .replace('Bmi', 'BMI')
           .replace('Lot', 'LoT')
           .replace('True', 'TMB % > 80')
           .replace('Afr', 'AFR').replace('Asj', 'ASJ').replace('Eas', 'EAS').replace('Eur', 'EUR').replace('Sas', 'SAS').replace('Adm', 'ADM')
           .replace('Ctla4', 'CTLA4')
        for ele in res.index
    ]

    res = res.reset_index(drop=True)
    sig_set = set(res.index[res['p'] < 0.05])
    res['y'] = np.arange(len(res))
    fig, ax = plt.subplots(figsize=(3, 1.5))

    for spine in ax.spines.values():
        spine.set_linewidth(LW)

    ax.hlines(
        y=res['y'], xmin=res['exp(coef) lower 95%'], xmax=res['exp(coef) upper 95%'],
        color=np.where(res['p'] < 0.05, c_attention, main_color), lw=LW, zorder=1,
    )
    sns.scatterplot(
        data=res, x='exp(coef)', y='y', hue='is_sig',
        palette={'P < 0.05': c_attention, 'NS': main_color},
        s=20, ax=ax, zorder=2,
    )
    ax.axvline(x=1, color='black', lw=LW, ls='--', zorder=0)
    ax.set_yticks(res['y'])
    ax.set_yticklabels(yticklabels, fontsize=tick_label_font_size)
    ax.invert_yaxis()
    for tick_label, idx in zip(ax.get_yticklabels(), res.index):
        if idx in sig_set:
            tick_label.set_color(c_attention)
        else:
            tick_label.set_color('black')
    if logscale:
        ax.set_xscale('log')
        ax.set_xticks([0.1, 0.5, 1, 2, 5, 10])
        ax.xaxis.set_minor_locator(mticker.NullLocator())
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:g}'))
        plt.setp(ax.get_xticklabels(), rotation=0, ha='center', fontsize=tick_label_font_size)
        for label in ax.get_xticklabels():
            label.set_clip_on(False)
    ax.tick_params(axis='x', which='major', bottom=True, length=4, width=LW)
    ax.tick_params(axis='x', which='minor', bottom=False, width=LW)
    ax.tick_params(axis='y', which='both', width=LW)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.2)
    ax.set_xlabel('Hazard Ratio', fontsize=axis_label_font_size)
    ax.set_ylabel('')
    ax.set_title('{} Position {}'.format(locus, pos), fontsize=axis_label_font_size)
    ax.legend(title='', loc='upper left', bbox_to_anchor=(1.05, 1), frameon=False)
    sns.despine(ax=ax, right=True, top=True)
    plt.tight_layout()

    plot_dir = os.path.join(figure_dir, exp_name, 'forest_plots_aa', f'{tox_type}_pos{pos}')
    os.makedirs(plot_dir, exist_ok=True)
    out_pdf = os.path.join(plot_dir, 'panel_e_{}_grade{:d}_pos{}_residues_forest.pdf'.format(tox_type, ae_grade, pos))
    out_png = os.path.join(plot_dir, 'panel_e_{}_grade{:d}_pos{}_residues_forest.png'.format(tox_type, ae_grade, pos))
    plt.savefig(out_pdf, dpi=1200)
    plt.savefig(out_png, dpi=1200)
    plt.close()
    return out_pdf


def plot_residue_forest(tox_type, ae_grade, pos, locus=LOCUS_OF_INTEREST, exp_name=EXP_NAME,
                         model_summary_dir=MODEL_SUMMARY_DIR, figure_dir=FIGURE_DIR):
    """Regenerate the residue-only forest plot for one specific
    (tox_type, ae_grade, pos), reading its Cox model summary"""
    summary_path = os.path.join(model_summary_dir, f'{tox_type}_grade{ae_grade}_pos{pos}_full_summary.csv')
    if not os.path.exists(summary_path):
        return None
    res = pd.read_csv(summary_path, index_col=0)
    return _forest_plot_from_summary(
        res=res, locus=locus, pos=pos, tox_type=tox_type, ae_grade=ae_grade, exp_name=exp_name,
        figure_dir=figure_dir,
    )


In [4]:
# Call plot_residue_forest on demand for the FDR<0.05 positions found in the
# saved omnibus results

for ae_grade in GRADES:
    res_path = os.path.join(COX_RES_DIR, f"{EXP_NAME}_{TOX_TYPE}_grade{ae_grade}_positional_cox_res.csv")
    if not os.path.exists(res_path):
        continue
    cur_results = pd.read_csv(res_path)
    sigs = cur_results[cur_results['fdr'] < 0.05]
    if len(sigs) == 0:
        continue
    for _, sig_row in sigs.iterrows():
        plot_residue_forest(tox_type=TOX_TYPE, ae_grade=ae_grade, pos=sig_row['ungapped_position'])
